In [ ]:
#Evaluations
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, ClusteringEvaluator, RegressionEvaluator

label_indexer = StringIndexer(inputCol="label", outputCol="labelIndex", handleInvalid="skip")
feature_cols = ["year","day","length","weight","count","looped","neighbors","income"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

acc_eval = MulticlassClassificationEvaluator(labelCol="labelIndex", predictionCol="prediction", metricName="accuracy")
f1_eval  = MulticlassClassificationEvaluator(labelCol="labelIndex", predictionCol="prediction", metricName="f1")
sil_eval = ClusteringEvaluator(featuresCol="features", metricName="silhouette")

rmse_eval = RegressionEvaluator(labelCol="labelIndex", predictionCol="prediction", metricName="rmse")
mae_eval  = RegressionEvaluator(labelCol="labelIndex", predictionCol="prediction", metricName="mae")
r2_eval   = RegressionEvaluator(labelCol="labelIndex", predictionCol="prediction", metricName="r2")

results = []

#Logistic Regression with CrossValidator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

lr = LogisticRegression(featuresCol="features", labelCol="labelIndex", maxIter=30)
pipe_lr = Pipeline(stages=[label_indexer, assembler, lr])

grid = (ParamGridBuilder()
        .addGrid(lr.regParam, [0.0, 0.1])
        .build())

cv = CrossValidator(estimator=pipe_lr,
                    estimatorParamMaps=grid,
                    evaluator=f1_eval,
                    numFolds=3,
                    parallelism=2,
                    seed=42)

cv_model = cv.fit(train_df)
lr_test  = cv_model.transform(test_df)

results.append(("LogReg_CV",
                float(rmse_eval.evaluate(lr_test)),
                float(mae_eval.evaluate(lr_test)),
                float(r2_eval.evaluate(lr_test))))

#Decision Tree
dt = DecisionTreeClassifier(featuresCol="features", labelCol="labelIndex", maxDepth=10)
pipe_dt = Pipeline(stages=[label_indexer, assembler, dt])

dt_model = pipe_dt.fit(train_df)
dt_test  = dt_model.transform(test_df)

results.append(("DecisionTree",
                float(rmse_eval.evaluate(dt_test)),
                float(mae_eval.evaluate(dt_test)),
                float(r2_eval.evaluate(dt_test))))

#Random Forest
rf = RandomForestClassifier(featuresCol="features", labelCol="labelIndex", numTrees=5, maxDepth=10)
pipe_rf = Pipeline(stages=[label_indexer, assembler, rf])

rf_model = pipe_rf.fit(train_df)
rf_test  = rf_model.transform(test_df)

results.append(("RandomForest",
                float(rmse_eval.evaluate(rf_test)),
                float(mae_eval.evaluate(rf_test)),
                float(r2_eval.evaluate(rf_test))))

#KMeans
km = KMeans(featuresCol="features", k=3, maxIter=10, seed=42)
pipe_km = Pipeline(stages=[assembler, km])

km_model = pipe_km.fit(train_df)
km_test  = km_model.transform(test_df)

cluster_count = km_test.select("prediction").distinct().count()

if cluster_count > 1:
    results.append(("KMeans",
                    float(rmse_eval.evaluate(km_test)),
                    float(mae_eval.evaluate(km_test)),
                    float(r2_eval.evaluate(km_test))))
else:
    results.append(("KMeans", None, None, None))

#Results Summary
print("\nResults Summary")
results_df = spark.createDataFrame(results, ["Model", "RMSE", "MAE", "R2"])
results_df.show(truncate=False)